In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings_model = HuggingFaceEmbeddings(
    model_name="./model/bge-base-zh-v1.5",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={
        "normalize_embeddings": True
    },  # 输出归一化向量，更适合余弦相似度计算
)

vectorstore = Chroma(
    embedding_function=embeddings_model,
    persist_directory="./vectorstore" #
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables.passthrough import RunnablePassthrough
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain.messages import HumanMessage, AIMessage
#LCEL方式来构建检索生成过程
retriever = vectorstore.as_retriever(
    search_type="similarity", # similarity或mmr
    search_kwargs={"k": 10} # 10~20
)

#ChatPromptTemplate
prompt = PromptTemplate(
    input_variables=["context", "question", "chat_history"],
    template="""
    你是一个专业的中文问答助手，擅长基于提供的资料回答用户问题。
    请仅根据以下背景资料回答问题，如无法找到答案，请直接回答“我不知道”。
    
    背景资料：{context}
    
    问题：{question}

    对话历史：{chat_history}

    回答：
    """,
)

def format_docs(docs):
    formatted_docs = "\n\n".join(doc.page_content for doc in docs)
    return formatted_docs

def format_chat_history(chat_history):
    if not chat_history: return "暂无对话历史"

    formatted_chat_history =[]
    for message in chat_history:
        if isinstance(message, HumanMessage):
            formatted_chat_history.append(f"用户：{message.content}")
        elif isinstance(message, AIMessage):
            formatted_chat_history.append(f"助手：{message.content}")
        else:
            formatted_chat_history.append(f"消息：{message.content}")
            
    return "\n".join(formatted_chat_history)

llm = init_chat_model("qwen3.7-plus",model_provider="openai")

rag_chain = (
    {
        "context": (lambda x: x["question"]) | retriever | format_docs, 
        "question": lambda x: x["question"],
        "chat_history": lambda x: format_chat_history(x["chat_history"]),
    }  
    | prompt 
    | (lambda x: print(x.text,end="") or x)
    | llm
    | StrOutputParser()
    )

In [4]:
chat_history = []
question1="中国科学院国家天文台2023年部门预算总额是多少"
result = rag_chain.invoke({
    "question":question1,
    "chat_history": chat_history
})
print(result)


    你是一个专业的中文问答助手，擅长基于提供的资料回答用户问题。
    请仅根据以下背景资料回答问题，如无法找到答案，请直接回答“我不知道”。

    背景资料：中 国 科 学 院 国家 天 文 台 2023 年 部 门 预算

二 、 中 国 科 学 院 国家 天 文 台 2023 年 部 门 预算

中 国 科 学 院 国 家 天 文 合 2023 年 初 部 门 预算 总 额 198,223.16 万 元

| app: 实施 中 国 科学 院 主管 部 门 及 代码 [173] 中 国 科 学 院 单位 国家 天 文 台 本 级 年 度 资金 总 额 : 16,158.00 项 目 资金 其 中 ， 财 政 拨款 16.158.00 ve 《万 元 ) 上 年 结 转 资金 0.00} (49) 其 他 资金 0.00 度 “ | 1 保障 设施 按 计划 实 现 安全 稳定 高 效 运行 和 日

、 中 国 科学 院 国家 天 文 台 基本 情况 1 (一 ) 单位 职责 ceececccccesseessessesseeseeseseesseeseesecsecseeseeseeneaneeneens 1 (二 ) 机 构 设 置 .es 2 、 中 国 科学 院 国 家 天 文 台 2023 年 部 门 预算 .ee 3 收 支 总 表

。 国家 天 文 台 本 级 年 度 资 金 总 额 : 120.00 项 目 资金 其 中 :财政 拨款 120.00 nn 《广元 上 年 结 转 资 金 000| (49) 其 他 资金 0.00 度 | 实时 太阳 图 像 的 高 速 采集 ,使 用 目前 较 成 熟 的 高 分 辩 率 图 像 重 构 方法 ， 克 服 大 气 滑 痢 | 流 的 影响 , 获得 接近 光学 衍射 极限

注 : 中 国 科学 院 国家 天 文 台 2023 年 没有 使 用 国有 资本 经 营 预算 安排 的 支出 。

。 中 国 科学 院 国家 天 文 全 部 门 预 算 既 包括 人 员 支 出 和 机 构 运 行 支出 ， 也 包括 竞争 性 经 费 、 人 才 引 进 与 培 养 、 科 研 条 件 及 后 勤 保 障 、 合 作 交 流 与 咨询 传播 、 科 普 活动 、 科研 设施 专项 运行 

In [5]:
chat_history.append(HumanMessage(content=question1))
chat_history.append(AIMessage(content=result))

In [6]:
# chat_history = []

# 指代消解
question2 = "该预算中，科学技术支出具体是多少？"
result2 = rag_chain.invoke({
    "question": question2,
    "chat_history": chat_history
})
print(result2)


    你是一个专业的中文问答助手，擅长基于提供的资料回答用户问题。
    请仅根据以下背景资料回答问题，如无法找到答案，请直接回答“我不知道”。

    背景资料：1. 科 学 技术 支出 (类 ): 反映 用 于 科学 技术 方面 的 支出 ， 中 国 科 学 院 预 算 中 主要 涉及 基础 研究 、 应 用 研究 、 技 术 研究 与 开发 、 科 技 条 件 与 服务 、 科 技 交流 与 合作 、 其 他 科学 技术 支出 等 款 级 支出 科目 。

2023 年 初 ， 科 学 技术 支出 预算 数 为 117,981.03 万 元 ; tt 会 保障 和 就 业 支 出 预算 数 为 1,930.68 万 元 ; 住房 保障 支出 预 算数 为 1.314.37 万 元 。

(S) 科 技 重大 项 目 : 反映 用 于 科技 重大 专项 等 有 关 经 费 支出 。

(4 ) 科技 交流 与 合作 : 反映 科技 交流 与 合作 等 方面 的 支 出 ,包括 为 提升 国家 科技 水 平 与 国外 政府 和 国际 组 织 作 和 研究、 科技 交流 方面 的 支出 ， 以 及 重 支出 等 rie AS 大 国际 科技 合作 专项

ihe 科目 名 称 — 本 年 一 般 公共 预算 支出 合计 基本 支出 项 目 支出 206 科学 技术 支出 54,561.72 10,362.47 44,199.25 20602 基础 研究 45,158.07 10,362.47 34,795.60 2060201 机 构 运行 10,362.47 10,362.47 _ 2060205 重大 科学 工程 16,158.00 _

(3 ) 科技 条 件 与 服务 : 反映 用 于 完善 科技 条 件 及 从 事 科 技 标准 、 计 量 和 检测 ， 科 技 数据 、 种 质 资 源 、 标 本 、 基 因 的 收集 、 加 工 处 理 和 服务 ， 科 技 文献 信息 资源 的 采集 、 保 存 、 加 工 和 服务 等 为 科技 活动 提供 基础 性 、 通用 性 服务 的 支出 。

科目 编码 科目 名 称 合计 基本 支出 项 目 支出 206 科学 技术 支出 143,981.03 16,362.47 127,618.56 2060

## 重述用户消息
在向大模型提问前，根据历史消息，将用户的问题重新生成。

In [7]:
# rephrase Prompt 模板
rephrase_prompt = PromptTemplate(
    input_variables=["chat_history", "question"],
    template="""
根据历史消息简要完善用户的问题，使其更加具体，实现指代消解，例如：'之前'、'该'等代词替换成具体的问题。只输出完善后的问题。

历史消息：[{chat_history}]

问题：{question}
""",
)

In [8]:
# 重述链条：根据历史和当前 query 生成更具体问题
rephrase_chain = (
    {
        "chat_history": lambda x: format_chat_history(x["chat_history"]),
        "question": lambda x: x["question"],
    }
    | rephrase_prompt
    | (lambda x: print(x.text, end="") or x)
    | llm
    | StrOutputParser()
    | (lambda x: print(f"===== 重述后的查询: {x}=====") or x)
)

In [9]:
chat_history = []
question1 = "中国科学院国家天文台2023年部门预算总额是多少"
result = rephrase_chain.invoke({
    "question": question1,
    "chat_history": chat_history
})

chat_history.append(HumanMessage(content=question1))
chat_history.append(AIMessage(content=result))


根据历史消息简要完善用户的问题，使其更加具体，实现指代消解，例如：'之前'、'该'等代词替换成具体的问题。只输出完善后的问题。

历史消息：[暂无对话历史]

问题：中国科学院国家天文台2023年部门预算总额是多少
===== 重述后的查询: 中国科学院国家天文台2023年部门预算总额是多少=====


In [10]:
#指代消解
question2 = "该预算中，科学技术支出具体是多少？" # 中国科学院国家天文台2023年部门预算中，科学技术支出具体是多少？
result2 = rephrase_chain.invoke({
    "question": question2,
    "chat_history": chat_history
})


根据历史消息简要完善用户的问题，使其更加具体，实现指代消解，例如：'之前'、'该'等代词替换成具体的问题。只输出完善后的问题。

历史消息：[用户：中国科学院国家天文台2023年部门预算总额是多少
助手：中国科学院国家天文台2023年部门预算总额是多少]

问题：该预算中，科学技术支出具体是多少？
===== 重述后的查询: 中国科学院国家天文台2023年部门预算中，科学技术支出具体是多少？=====


In [11]:
rag_chain = (
    {
        "context": lambda x: format_docs(
            retriever.invoke(
                #需要重新构造问题
                rephrase_chain.invoke({
                    "question": x["question"],
                    "chat_history": x["chat_history"]})
        )), 
        "question": lambda x: x["question"],
        "chat_history": lambda x: format_chat_history(x["chat_history"]),
    }  
    | prompt 
    | (lambda x: print(x.text, end="") or x)
    | llm
    | StrOutputParser()
    )

In [ ]:
chat_history = []
question1 = "中国科学院国家天文台2023年部门预算总额是多少"
result = rag_chain.invoke({
    "question": question1,
    "chat_history": chat_history
})
print(result)


根据历史消息简要完善用户的问题，使其更加具体，实现指代消解，例如：'之前'、'该'等代词替换成具体的问题。只输出完善后的问题。

历史消息：[暂无对话历史]

问题：中国科学院国家天文台2023年部门预算总额是多少
===== 重述后的查询: 中国科学院国家天文台2023年部门预算总额是多少=====

    你是一个专业的中文问答助手，擅长基于提供的资料回答用户问题。
    请仅根据以下背景资料回答问题，如无法找到答案，请直接回答“我不知道”。

    背景资料：中 国 科 学 院 国家 天 文 台 2023 年 部 门 预算

二 、 中 国 科 学 院 国家 天 文 台 2023 年 部 门 预算

中 国 科 学 院 国 家 天 文 合 2023 年 初 部 门 预算 总 额 198,223.16 万 元

| app: 实施 中 国 科学 院 主管 部 门 及 代码 [173] 中 国 科 学 院 单位 国家 天 文 台 本 级 年 度 资金 总 额 : 16,158.00 项 目 资金 其 中 ， 财 政 拨款 16.158.00 ve 《万 元 ) 上 年 结 转 资金 0.00} (49) 其 他 资金 0.00 度 “ | 1 保障 设施 按 计划 实 现 安全 稳定 高 效 运行 和 日

、 中 国 科学 院 国家 天 文 台 基本 情况 1 (一 ) 单位 职责 ceececccccesseessessesseeseeseseesseeseesecsecseeseeseeneaneeneens 1 (二 ) 机 构 设 置 .es 2 、 中 国 科学 院 国 家 天 文 台 2023 年 部 门 预算 .ee 3 收 支 总 表

。 国家 天 文 台 本 级 年 度 资 金 总 额 : 120.00 项 目 资金 其 中 :财政 拨款 120.00 nn 《广元 上 年 结 转 资 金 000| (49) 其 他 资金 0.00 度 | 实时 太阳 图 像 的 高 速 采集 ,使 用 目前 较 成 熟 的 高 分 辩 率 图 像 重 构 方法 ， 克 服 大 气 滑 痢 | 流 的 影响 , 获得 接近 光学 衍射 极限

注 : 中 国 科学 院 国家 天 文 台 2023 年 没有 使 用 国有 资本 经 营 预算 安排 的 

In [13]:
chat_history.append(HumanMessage(content=question1))
chat_history.append(AIMessage(content=result))

In [16]:
question2 = "该预算中，科学技术支出具体是多少？" # 中国科学院国家天文台2023年部门预算中，科学技术支出具体是多少？
result2 = rag_chain.invoke({
    "question": question2,
    "chat_history": chat_history
})

chat_history.append(HumanMessage(content=question2))
chat_history.append(AIMessage(content=result2))

print(result2)



根据历史消息简要完善用户的问题，使其更加具体，实现指代消解，例如：'之前'、'该'等代词替换成具体的问题。只输出完善后的问题。

历史消息：[用户：中国科学院国家天文台2023年部门预算总额是多少
助手：中国科学院国家天文台2023年部门预算总额是198,223.16万元。
用户：该预算中，科学技术支出具体是多少？
助手：根据背景资料，2023年初科学技术支出预算数为117,981.03万元。]

问题：该预算中，科学技术支出具体是多少？
===== 重述后的查询: 中国科学院国家天文台2023年部门预算中，科学技术支出具体是多少？=====

    你是一个专业的中文问答助手，擅长基于提供的资料回答用户问题。
    请仅根据以下背景资料回答问题，如无法找到答案，请直接回答“我不知道”。

    背景资料：中 国 科 学 院 国家 天 文 台 2023 年 部 门 预算

二 、 中 国 科 学 院 国家 天 文 台 2023 年 部 门 预算

中 国 科 学 院 国 家 天 文 合 2023 年 初 部 门 预算 总 额 198,223.16 万 元

注 : 中 国 科学 院 国家 天 文 台 2023 年 没有 使 用 国有 资本 经 营 预算 安排 的 支出 。

2023 年 初 ， 科 学 技术 支出 预算 数 为 117,981.03 万 元 ; tt 会 保障 和 就 业 支 出 预算 数 为 1,930.68 万 元 ; 住房 保障 支出 预 算数 为 1.314.37 万 元 。

| app: 实施 中 国 科学 院 主管 部 门 及 代码 [173] 中 国 科 学 院 单位 国家 天 文 台 本 级 年 度 资金 总 额 : 16,158.00 项 目 资金 其 中 ， 财 政 拨款 16.158.00 ve 《万 元 ) 上 年 结 转 资金 0.00} (49) 其 他 资金 0.00 度 “ | 1 保障 设施 按 计划 实 现 安全 稳定 高 效 运行 和 日

2023 年 工作 总 体 思路 是 : 深入 学 习 贯 彻 党 的 二 十 大 精 神 ， 落 实 党 的 全 面 领导 ， 落 实 党 中 央 、 国 务 院 和 院 党 组 重大 决策 部 署 ， 面 向 国家 重大 需求 、 面 向 国际 天 文科 技 

## RAG完整流程
1.加载文档(pdf)
    重点关注:pdf文档的加载,图片| 表格
2.数据清洗

3.切片:递归滑动窗口切片
    chunk_size
    chunk_overlap

4.向量存储:Embedding模型-》BEG

5.相似度检索 检索方式:similarity, mmr topk > 20

6.多轮对话中要做问题重述